# Fine-tune V-JEPA2 (SSv2) on SAILS rmm_type

This notebook mirrors the structure of `scripts/notebook_finetuning.ipynb` but targets the SAILS clips generated from the CSV folds. It uses the `rmm_type` column as labels and assumes the clips live under `/orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips/<csv_stem>/<segment_id>.mp4` as produced by `dataprep/create_clip_segments.py` with deduplication.

## Setup
The environment should already have `torch`, `numpy`, `decord`, and `transformers` (>=4.44) installed. Adjust the paths below if your clip or CSV locations differ.


In [1]:
from pathlib import Path
import csv
from functools import partial
from typing import List, Dict

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader, cpu

from transformers import VJEPA2ForVideoClassification, VJEPA2VideoProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
csv_dir = Path("/orcd/data/satra/001/users/brukew/actreg/dataprep/cv_folds")
clips_root = Path("/orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips")

# Pick which folds to use for train/val
train_csvs = ["fold_0_train.csv", "fold_1_train.csv", "fold_2_train.csv"]
val_csvs = ["fold_0_val.csv", "fold_1_val.csv", "fold_2_val.csv"]
cur_train, cur_val = [train_csvs[0]], [val_csvs[0]]

model_id = "facebook/vjepa2-vitl-fpc16-256-ssv2"

proj_name = "vjepa-rmm"

csv_dir, clips_root, model_id


(PosixPath('/orcd/data/satra/001/users/brukew/actreg/dataprep/cv_folds'),
 PosixPath('/orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips'),
 'facebook/vjepa2-vitl-fpc16-256-ssv2')

## Build split manifests from CSV + clip folders
Each CSV row should have `segment_id` (or `segment_global_id`), `rmm_type`, and the clip should exist in `<clips_root>/<csv_stem>/<segment_id>.mp4`. This cell collects train/val records and checks for missing clips.

In [2]:
def load_split(csv_names: List[str]) -> List[Dict]:
    records = []
    missing = []
    for name in csv_names:
        csv_path = csv_dir / name
        stem = csv_path.stem
        with csv_path.open("r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fieldnames = [fn.strip().replace("\ufeff", "") for fn in reader.fieldnames]
            reader.fieldnames = fieldnames
            for row in reader:
                seg = row.get("segment_id") or row.get("segment_global_id")
                label = row.get("rmm_type")
                if not seg or label is None:
                    continue
                clip_path = clips_root / stem / f"{seg}.mp4"
                if not clip_path.exists():
                    missing.append((seg, clip_path))
                    continue
                records.append({"segment_id": seg, "label": label, "clip": clip_path})
    print(f"Loaded {len(records)} records from {len(csv_names)} CSVs; missing clips: {len(missing)}")
    if missing:
        print("Example missing:", missing[:3])
    return records

train_records = load_split(cur_train)
val_records = load_split(cur_val)
all_labels = sorted({r["label"] for r in train_records})
label2id = {lbl: i for i, lbl in enumerate(all_labels)}
id2label = {i: lbl for lbl, i in label2id.items()}
len(train_records), len(val_records), label2id

Loaded 349 records from 1 CSVs; missing clips: 0
Loaded 175 records from 1 CSVs; missing clips: 0


(349,
 175,
 {'hands flapping': 0,
  'jumping': 1,
  'one hand flap': 2,
  'rocking': 3,
  'spinning': 4})

In [3]:
processor = VJEPA2VideoProcessor.from_pretrained(model_id)

frames_per_clip = (
    getattr(processor, "num_frames", None)
    or getattr(getattr(processor, "image_processor", processor), "num_frames", None)
    or getattr(getattr(processor, "feature_extractor", processor), "num_frames", None)
    or getattr(getattr(processor, "config", {}), "num_frames", None)
    or getattr(getattr(processor, "config", {}), "frames_per_clip", None)
    or 16
)
frames_per_clip = 32
frames_per_clip


32

## Dataset + DataLoaders
We sample frames with Decord using simple even spacing (one clip per video) and let the processor handle normalization/resizing.


In [4]:
class RMMDataset(Dataset):
    def __init__(self, records, label2id, frames_per_clip):
        self.records = records
        self.label2id = label2id
        self.frames_per_clip = frames_per_clip

    def __len__(self):
        return len(self.records)

    def _sample_indices(self, vr):
        total = len(vr)
        if total <= 0:
            return np.zeros(self.frames_per_clip, dtype=np.int64)
        return np.round(np.linspace(0, total - 1, self.frames_per_clip)).astype("int64")

    def __getitem__(self, idx):
        rec = self.records[idx]
        try:
            vr = VideoReader(str(rec["clip"]), ctx=cpu(0))
            indices = self._sample_indices(vr)
            frames = vr.get_batch(indices).asnumpy()  # (T, H, W, C) uint8
        except Exception as e:
            print(f"[bad clip] {rec['clip']}: {e}")
            return None
        label_id = self.label2id[rec["label"]]
        return frames, label_id


def collate_fn(samples, processor):
    samples = [s for s in samples if s is not None]
    if not samples:
        return None, None
    frame_batches, labels = zip(*samples)
    inputs = processor(list(frame_batches), return_tensors="pt")
    labels = torch.tensor(labels)
    return inputs, labels

train_ds = RMMDataset(train_records, label2id, frames_per_clip)
val_ds = RMMDataset(val_records, label2id, frames_per_clip)

batch_size = 1
num_workers = 8

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=partial(collate_fn, processor=processor),
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=num_workers > 0,
    prefetch_factor=2 if num_workers > 0 else None,
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=partial(collate_fn, processor=processor),
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=num_workers > 0,
    prefetch_factor=2 if num_workers > 0 else None,
)

batch0 = next(iter(train_loader))
batch0[0] if batch0[0] is None else batch0[0]['pixel_values_videos'].shape, len(train_ds), len(val_ds)


(torch.Size([1, 32, 3, 256, 256]), 349, 175)

## Initialize model for classification (rmm_type head)
We freeze the V-JEPA backbone and train only the classification head.

In [5]:
model = VJEPA2ForVideoClassification.from_pretrained(
    model_id,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to(device)

for param in model.vjepa2.parameters():
    param.requires_grad = False

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable, lr=1e-5)
num_epochs = 10
accumulation_steps = 4

model.config

Some weights of VJEPA2ForVideoClassification were not initialized from the model checkpoint at facebook/vjepa2-vitl-fpc16-256-ssv2 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([174]) in the checkpoint and torch.Size([5]) in the model instantiated
- classifier.weight: found shape torch.Size([174, 1024]) in the checkpoint and torch.Size([5, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


VJEPA2Config {
  "architectures": [
    "VJEPA2ForVideoClassification"
  ],
  "attention_dropout": 0.0,
  "attention_probs_dropout_prob": 0.0,
  "crop_size": 256,
  "drop_path_rate": 0.0,
  "dtype": "float32",
  "frames_per_clip": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 1024,
  "id2label": {
    "0": "hands flapping",
    "1": "jumping",
    "2": "one hand flap",
    "3": "rocking",
    "4": "spinning"
  },
  "image_size": 256,
  "in_chans": 3,
  "initializer_range": 0.02,
  "label2id": {
    "hands flapping": 0,
    "jumping": 1,
    "one hand flap": 2,
    "rocking": 3,
    "spinning": 4
  },
  "layer_norm_eps": 1e-06,
  "mlp_ratio": 4,
  "model_type": "vjepa2",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "num_pooler_layers": 3,
  "patch_size": 16,
  "pred_hidden_size": 384,
  "pred_mlp_ratio": 4.0,
  "pred_num_attention_heads": 12,
  "pred_num_hidden_layers": 12,
  "pred_num_mask_tokens": 10,
  "pred_zero_init_mask_tokens": true,
  "q

## Training + simple eval
Gradient accumulation simulates a larger batch size. Evaluation reports accuracy on the validation set.

In [6]:
import wandb
run_name = f"base-whole-video-{frames_per_clip}fr"
wandb.init(project=proj_name, name=run_name, config={"lr": 1e-5, "batch_size": 1, "frames": frames_per_clip})

wandb: Currently logged in as: brukew (brukew-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
def evaluate(loader, model, device):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            if inputs is None or labels is None:
                continue
            labels = labels.to(device)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = logits.argmax(-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / max(total, 1)
    return acc, all_preds, all_labels


def class_names_from_id2label(id2label):
    return [id2label[i] for i in range(len(id2label))]


for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()
    running_loss = 0.0
    num_batches = 0

    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{num_epochs}")
    print("=" * 60)

    for step, (inputs, labels) in enumerate(train_loader, start=1):
        if inputs is None or labels is None:
            continue

        labels = labels.to(device)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss / accumulation_steps
        loss.backward()
        running_loss += loss.item()
        num_batches += 1

        if step % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        if step % 50 == 0:
            avg_loss = running_loss / num_batches * accumulation_steps
            print(f"  Step {step}/{len(train_loader)} | Loss: {avg_loss:.4f}")

        wandb.log({"train/loss": loss, "epoch": epoch, "step": step})

    if num_batches % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = running_loss / max(num_batches, 1) * accumulation_steps
    val_acc, val_preds, val_labels = evaluate(val_loader, model, device)
    wandb.log({"val/acc": val_acc, "epoch": epoch})

    if val_labels:
        cm = wandb.plot.confusion_matrix(
            preds=val_preds,
            y_true=val_labels,
            class_names=class_names_from_id2label(id2label),
        )
        wandb.log({"val/conf_mat": cm, "epoch": epoch})

    print(f"\n>>> Epoch {epoch} complete: avg_loss={avg_loss:.4f}, val_acc={val_acc:.3f}")

output_dir = Path("runs/vjepa2_rmm_type")
output_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
output_dir
wandb.finish()


Epoch 1/10
  Step 50/349 | Loss: 1.6686
  Step 100/349 | Loss: 1.6655
  Step 150/349 | Loss: 1.6582
  Step 200/349 | Loss: 1.6367
  Step 250/349 | Loss: 1.6206
  Step 300/349 | Loss: 1.5761
[bad clip] /orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips/fold_0_val/L4K0P3T4Q8_36_month_185_0_0.mp4: [21:18:44] /github/workspace/src/video/video_reader.cc:151: Check failed: st_nb >= 0 (-1381258232 vs. 0) ERROR cannot find video stream with wanted index: -1

>>> Epoch 1 complete: avg_loss=1.5500, val_acc=0.408

Epoch 2/10
  Step 50/349 | Loss: 1.3244
  Step 100/349 | Loss: 1.3399
  Step 150/349 | Loss: 1.3210
  Step 200/349 | Loss: 1.3562
  Step 250/349 | Loss: 1.3630
  Step 300/349 | Loss: 1.3589
[bad clip] /orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips/fold_0_val/L4K0P3T4Q8_36_month_185_0_0.mp4: [21:21:54] /github/workspace/src/video/video_reader.cc:151: Check failed: st_nb >= 0 (-1381258232 vs. 0) ERROR cannot find video stream with wanted index: -1

>>> Epoch 2 c

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇████
step,▁▂▃▇▇▂▃▅▆▇█▂▃▄▅█▄▄▅▆▅▂▆▂▃▆▆▆▁▃▅▆▆▇█▂▂▂▃▆
train/loss,▇▆▃▆▄▄█▇█▄▂▂█▃▄▆▆▆▅▂▆▄▂▃▂▂▄▂▂▃▃▂▂▂▁▃▃▃▁▂
val/acc,▁▃▅▄▅▂▂▅▄█
epoch,10
step,349
train/loss,0.04646
val/acc,0.48276


## Next steps
- Push to Hub with `model.push_to_hub(...)` if desired.
- Add more augmentations/regularization as needed for better performance.
- Increase batch size on larger GPUs or adjust `accumulation_steps`.